In [7]:
import pandas as pd
import numpy as np
from scipy.special import eval_legendre
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

# =====================================================
# LOAD DATA
# =====================================================

df_raw = pd.read_csv("raw_orbit_data.csv")

mu = 3.986004418e14
Re = 6378137.0

output_dir = Path("lifted_dictionaries")
output_dir.mkdir(exist_ok=True)

# =====================================================
# EXPERIMENT CONFIGS
# =====================================================

configs = [
    {"name": "base",  "legendre": False, "fourier": False, "degree": 0},
    {"name": "leg2",  "legendre": True,  "fourier": False, "degree": 2},
    {"name": "leg4",  "legendre": True,  "fourier": False, "degree": 4},
    {"name": "leg6",  "legendre": True,  "fourier": False, "degree": 6},

    {"name": "four2", "legendre": False, "fourier": True,  "degree": 2},
    {"name": "four4", "legendre": False, "fourier": True,  "degree": 4},
    {"name": "four6", "legendre": False, "fourier": True,  "degree": 6},

    {"name": "both2", "legendre": True,  "fourier": True,  "degree": 2},
    {"name": "both4", "legendre": True,  "fourier": True,  "degree": 4},
    {"name": "both6", "legendre": True,  "fourier": True,  "degree": 6},
]

metadata_rows = []

# =====================================================
# GENERATE ALL DICTIONARIES
# =====================================================

for cfg in configs:

    print(f"Generating {cfg['name']}")

    df_proc = df_raw.copy()

    # ---------------------------------------------
    # Physics-informed features
    # ---------------------------------------------

    lifted_features = {}

    lifted_features["mean_motion_n"] = np.sqrt(
        mu / (df_proc["a"] ** 3)
    )

    p = df_proc["a"] * (1 - df_proc["e"] ** 2)

    J2_factor = (Re / p) ** 2

    lifted_features["J2_raan_term"] = (
        J2_factor * np.cos(df_proc["i"])
    )

    lifted_features["J2_omega_term"] = (
        J2_factor * (
            4 - 5 * (np.sin(df_proc["i"]) ** 2)
        )
    )

    # ---------------------------------------------
    # Scale a,e
    # ---------------------------------------------

    scaler_ae = MinMaxScaler(feature_range=(-1, 1))

    df_proc[["a", "e"]] = scaler_ae.fit_transform(
        df_proc[["a", "e"]]
    )

    df_phys = pd.DataFrame(lifted_features)

    scaler_phys = MinMaxScaler(feature_range=(-1, 1))

    df_phys[df_phys.columns] = scaler_phys.fit_transform(
        df_phys
    )

    # ---------------------------------------------
    # Dictionary
    # ---------------------------------------------

    math_features = {}

    if cfg["legendre"]:

        for col in ["a", "e"]:

            x = df_proc[col].values

            for degree in range(
                2,
                cfg["degree"] + 1
            ):

                math_features[
                    f"{col}_P{degree}"
                ] = eval_legendre(
                    degree,
                    x
                )

    if cfg["fourier"]:

        for angle_col in [
            "i",
            "omega",
            "raan",
            "m"
        ]:

            angle = df_proc[angle_col].values

            for k in range(
                1,
                cfg["degree"] + 1
            ):

                math_features[
                    f"sin_{k}_{angle_col}"
                ] = np.sin(k * angle)

                math_features[
                    f"cos_{k}_{angle_col}"
                ] = np.cos(k * angle)

    df_math = pd.DataFrame(math_features)
    
    df_final = pd.concat(
        [
            df_proc,
            df_phys,
            df_math
        ],
        axis=1
    )
    
    filename = (
        output_dir /
        f"lifted_{cfg['name']}.csv"
    )
    
    with open(filename, "w") as f:
    
        f.write(f"# dictionary={cfg['name']}\n")
        f.write(f"# legendre={cfg['legendre']}\n")
        f.write(f"# fourier={cfg['fourier']}\n")
        f.write(f"# degree={cfg['degree']}\n")
        f.write(f"# n_features={len(df_final.columns)}\n")
    
        df_final.to_csv(
            f,
            index=False
        )
print("done")

Generating base
Generating leg2
Generating leg4
Generating leg6
Generating four2
Generating four4
Generating four6
Generating both2
Generating both4
Generating both6
done
